# Notebook 9 — Final Integration + XAI Dashboard + LLM + Human-in-the-Loop

This notebook demonstrates the complete Apple Plant Disease XAI pipeline using **four exact rows from Notebook 4's processed `test.pt` split**—not files from `test_images/`:

- **LOW-TRUST SAMPLE 1:** row 1060
- **LOW-TRUST SAMPLE 2:** row 820
- **LOW-TRUST SAMPLE 3:** row 786
- **HIGH-TRUST SAMPLE:** row 364

The notebook loads all four rows, previews them, and calculates ResNet50 prediction/trust metrics for all four. The full Grad-CAM → RAG → Qwen → human-review workflow runs on the case selected by `DEMO_CASE_POSITION` in Cell 3. Change that value from 0 to 1, 2, or 3 and rerun from Cell 7 onward to demonstrate another case.

**Image → ResNet50 → Prediction/Probabilities → Trust/Uncertainty → Grad-CAM → RAG → Qwen Grounded Explanation → Human Review → Persistent Feedback Memory**

Important:
- ResNet50 is the classifier.
- Grad-CAM highlights image regions that influenced the model prediction; it does not prove disease.
- RAG supplies disease knowledge.
- Qwen is an explanation model, not the classifier.
- Human feedback is stored persistently.
- This notebook does not retrain ResNet50 from feedback.


In [ ]:
# Cell 1 — Imports and environment
import os
import json
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 2 — Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)


In [ ]:
# Cell 3 — Project configuration
PROJECT_DIR = Path("/content/drive/MyDrive/XAI_Project")
RESULTS_DIR = PROJECT_DIR / "results"
FINAL_DIR = RESULTS_DIR / "final_integration"
FEEDBACK_DIR = RESULTS_DIR / "human_feedback"

# Notebook 4 processed split and model artifacts.
# Change these two paths only if Notebook 4 saved them elsewhere.
DATA_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "saved_models"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4

FINAL_DIR.mkdir(parents=True, exist_ok=True)
FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Apple Healthy",
    "Apple Scab",
    "Apple Black Rot",
    "Apple Cedar Apple Rust",
]

# Exact rows from the trust report, in processed/test.pt order.
DEMO_CASES = [
    {"demo_case": "LOW-TRUST SAMPLE 1", "image_index": 1060, "reported_trust_group": "Low"},
    {"demo_case": "LOW-TRUST SAMPLE 2", "image_index": 820,  "reported_trust_group": "Low"},
    {"demo_case": "LOW-TRUST SAMPLE 3", "image_index": 786,  "reported_trust_group": "Low"},
    {"demo_case": "HIGH-TRUST SAMPLE",  "image_index": 364,  "reported_trust_group": "High"},
]
# 0 = Low-trust 1, 1 = Low-trust 2, 2 = Low-trust 3, 3 = High-trust.
# The complete pipeline below runs for this selected case; all four are compared in Cell 7.
DEMO_CASE_POSITION = 0

print("Project:", PROJECT_DIR)
print("Processed test split:", DATA_DIR / "test.pt")
print("Model directory:", MODEL_DIR)
print("Device:", DEVICE)


In [ ]:
# Cell 4 — Artifact check
print("=" * 70)
print("PROJECT ARTIFACT CHECK")
print("=" * 70)
for p in [PROJECT_DIR, RESULTS_DIR, FINAL_DIR, FEEDBACK_DIR, DATA_DIR, MODEL_DIR]:
    print(("✓" if p.exists() else "✗"), p)

print("\nRelevant model/result files:")
for root in [MODEL_DIR, PROJECT_DIR / "saved_models", RESULTS_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file() and p.suffix.lower() in {".pth", ".pt", ".keras", ".csv", ".json", ".jsonl"}:
                print(" ", p)


In [ ]:
# Cell 5 — Find the same ResNet50 checkpoint candidates as Notebook 4
def find_checkpoint():
    candidates = [
        MODEL_DIR / "best_overall_model.pt",
        MODEL_DIR / "resnet50_best.pt",
        MODEL_DIR / "best_resnet50.pt",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "No ResNet50 checkpoint found. Checked: " + ", ".join(map(str, candidates))
    )

CHECKPOINT_PATH = find_checkpoint()
print("Using checkpoint:", CHECKPOINT_PATH)


In [ ]:
# Cell 6 — Reconstruct and load the four-class ResNet50 model
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        state_dict = checkpoint
else:
    state_dict = checkpoint

state_dict = {
    (k.replace("module.", "", 1) if k.startswith("module.") else k): v
    for k, v in state_dict.items()
}
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys or unexpected_keys:
    print("Checkpoint key warning | missing:", missing_keys, "| unexpected:", unexpected_keys)

model = model.to(DEVICE).eval()
print("ResNet50 ready on:", DEVICE)


In [ ]:
# Cell 7 — Load the four exact processed test rows from the trust report
# Images are extracted by row index from processed/test.pt; no external image paths are used.
def load_processed_split(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Processed split not found: {path}. Run Notebook 4 data preparation first."
        )
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(path, map_location="cpu")

    if isinstance(obj, (tuple, list)) and len(obj) == 2:
        images, labels = obj
    elif isinstance(obj, dict):
        x_key = next((k for k in ["images", "X", "data", "inputs"] if k in obj), None)
        y_key = next((k for k in ["labels", "y", "targets"] if k in obj), None)
        if x_key is None or y_key is None:
            raise ValueError(f"Could not identify image/label keys in {path}; keys={list(obj)}")
        images, labels = obj[x_key], obj[y_key]
    else:
        raise ValueError(f"Unsupported processed split format: {type(obj)}")

    return torch.as_tensor(images).float(), torch.as_tensor(labels).long()


test_images, test_labels = load_processed_split("test.pt")
if test_images.ndim != 4:
    raise ValueError(f"Expected test tensor shape [N,C,H,W], got {tuple(test_images.shape)}")
if test_images.shape[1] != 3:
    raise ValueError(f"Expected 3-channel test tensor, got shape {tuple(test_images.shape)}")
if len(test_labels) != len(test_images):
    raise ValueError(f"Image/label count mismatch: {len(test_images)} images vs {len(test_labels)} labels")

invalid = [int(c["image_index"]) for c in DEMO_CASES
           if int(c["image_index"]) < 0 or int(c["image_index"]) >= len(test_images)]
if invalid:
    raise IndexError(
        f"Trust report indices outside processed/test.pt: {invalid}; test rows={len(test_images)}"
    )
if not 0 <= int(DEMO_CASE_POSITION) < len(DEMO_CASES):
    raise IndexError(f"DEMO_CASE_POSITION must be 0..{len(DEMO_CASES)-1}")

# Reconstruct the report's exact index mapping and display all four selected inputs.
selected = pd.DataFrame(DEMO_CASES)
selected["image_ref"] = selected["image_index"].map(
    lambda i: f"processed/test.pt[index={int(i)}]"
)
display(selected[["demo_case", "image_index", "image_ref", "reported_trust_group"]])


def tensor_to_display_image(tensor):
    """Convert a stored CHW tensor to a display image without changing model input."""
    t = tensor.detach().cpu().clone()
    if t.min().item() < 0 or t.max().item() > 1:
        # Reverse ImageNet normalization for display only, if present.
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        t = t * std + mean
    t = t.clamp(0, 1)
    return Image.fromarray((t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))


# Preview the exact four rows and calculate prediction/trust metrics for all four.
# These are the model's computed trust scores; the report's Low/High tags are preserved separately.
comparison_rows = []
fig, axes = plt.subplots(1, len(DEMO_CASES), figsize=(16, 4))
for ax, case in zip(axes, DEMO_CASES):
    idx = int(case["image_index"])
    ax.imshow(tensor_to_display_image(test_images[idx]))
    ax.axis("off")
    ax.set_title(f"{case['demo_case']}\nindex={idx}")

    x = test_images[idx].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(model(x), dim=1)[0].detach().cpu().numpy()
    pred = int(np.argmax(p))
    conf = float(p[pred])
    ent = float(-np.sum(p * np.log(np.clip(p, 1e-12, 1.0))) / np.log(len(CLASS_NAMES)))
    sorted_p = np.sort(p)[::-1]
    margin_value = float(sorted_p[0] - sorted_p[1])
    score = float(np.clip(0.5 * conf + 0.3 * (1.0 - ent) + 0.2 * margin_value, 0.0, 1.0))
    level = "High" if score >= 0.80 else ("Medium" if score >= 0.60 else "Low")
    comparison_rows.append({
        "demo_case": case["demo_case"],
        "image_index": idx,
        "reported_trust_group": case["reported_trust_group"],
        "predicted_class": CLASS_NAMES[pred],
        "confidence": conf,
        "entropy": ent,
        "margin": margin_value,
        "computed_trust_score": score,
        "computed_trust_level": level,
        "human_review_required": bool(score < 0.70 or conf < 0.80 or ent > 0.50 or margin_value < 0.20),
    })
plt.tight_layout()
plt.show()

comparison_df = pd.DataFrame(comparison_rows)
print("ResNet50 prediction/trust comparison for the four specified test rows:")
display(comparison_df)

# Choose which one of the four cases is fed to the full Grad-CAM + RAG + Qwen + HITL pipeline.
active_case = DEMO_CASES[int(DEMO_CASE_POSITION)]
DEMO_IMAGE_INDEX = int(active_case["image_index"])
input_tensor = test_images[DEMO_IMAGE_INDEX].unsqueeze(0).to(DEVICE)
true_label_id = int(test_labels[DEMO_IMAGE_INDEX].item())
IMAGE_PATH = f"processed/test.pt[index={DEMO_IMAGE_INDEX}]"
image = tensor_to_display_image(test_images[DEMO_IMAGE_INDEX])
IMAGE_SIZE = int(input_tensor.shape[-1])

print("\nActive full-pipeline demo:", active_case["demo_case"])
print("Image reference:", IMAGE_PATH)
print("Test tensor:", tuple(test_images.shape), "| labels:", tuple(test_labels.shape))
print("Selected row:", DEMO_IMAGE_INDEX, "| ground-truth label id:", true_label_id)
print("Model input tensor:", tuple(input_tensor.shape))
print("Loaded ResNet50 checkpoint:", CHECKPOINT_PATH)


In [ ]:
# Cell 8 — Prediction + uncertainty + trust
with torch.no_grad():
    logits = model(input_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]

pred_idx = int(torch.argmax(probabilities).item())
pred_class = CLASS_NAMES[pred_idx]
probs = probabilities.detach().cpu().numpy()

confidence = float(probs[pred_idx])
entropy = float(-np.sum(probs * np.log(np.clip(probs, 1e-12, 1.0))) / np.log(len(CLASS_NAMES)))

sorted_probs = np.sort(probs)[::-1]
margin = float(sorted_probs[0] - sorted_probs[1])

trust_score = float(np.clip(
    0.5 * confidence + 0.3 * (1.0 - entropy) + 0.2 * margin,
    0.0, 1.0
))

trust_level = "High" if trust_score >= 0.80 else ("Medium" if trust_score >= 0.60 else "Low")

human_review_required = (
    trust_score < 0.70
    or confidence < 0.80
    or entropy > 0.50
    or margin < 0.20
)

print("=" * 70)
print("MODEL PREDICTION")
print("=" * 70)
print("Prediction:", pred_class)
print(f"Confidence: {confidence:.4f}")
print(f"Entropy:    {entropy:.4f}")
print(f"Margin:     {margin:.4f}")
print(f"Trust:      {trust_score:.4f}")
print("Trust level:", trust_level)
print("Human review:", "REQUIRED" if human_review_required else "NOT REQUIRED")

print("\nCLASS PROBABILITIES")
for i, p in enumerate(probs):
    print(f"{CLASS_NAMES[i]:35s}: {p:.4f} ({p*100:.2f}%)")


In [ ]:
# Cell 9 — Grad-CAM
target_layer = model.layer4[-1]
activations, gradients = [], []

def forward_hook(module, inp, output):
    activations.append(output)

def backward_hook(module, grad_input, grad_output):
    gradients.append(grad_output[0])

fh = target_layer.register_forward_hook(forward_hook)
bh = target_layer.register_full_backward_hook(backward_hook)

model.zero_grad(set_to_none=True)
logits_gc = model(input_tensor)
logits_gc[0, pred_idx].backward()

fh.remove()
bh.remove()

acts = activations[0].detach()
grads = gradients[0].detach()
weights = grads.mean(dim=(2, 3), keepdim=True)

cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
cam = F.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
cam = cam[0, 0].cpu().numpy()
cam -= cam.min()
if cam.max() > 0:
    cam /= cam.max()

high_activation_area = float((cam >= 0.70).mean() * 100.0)

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.imshow(cam, cmap="jet", alpha=0.45)
plt.axis("off")
plt.title(f"Grad-CAM — {pred_class}")
plt.tight_layout()

gradcam_path = FINAL_DIR / "latest_gradcam.png"
plt.savefig(gradcam_path, dpi=180, bbox_inches="tight")
plt.show()

print("Grad-CAM target layer: layer4[-1]")
print(f"High activation area: {high_activation_area:.2f}%")
print("Saved:", gradcam_path)


In [ ]:
# Cell 10 — RAG knowledge
LOCAL_KNOWLEDGE = {
    "Apple Scab": [
        "Apple scab leaf spots are commonly olive-green to brown and can become dark brown to black as they age. Heavily affected leaves can yellow and drop.",
        "Apple scab overwinters in infected fallen leaves. Spring spores can infect developing leaves, flowers, fruit, and twigs when moisture conditions are favorable.",
        "Management of apple scab includes practices that improve airflow and leaf drying, removal of infected fallen leaves where appropriate, resistant varieties, and properly timed disease-control programs.",
    ],
    "Apple Black Rot": [
        "Apple black rot can produce circular brown to black leaf spots and dark fruit lesions.",
        "Black rot symptoms may include frog-eye leaf spots and fruit rot. Disease development is associated with infected plant material and favorable environmental conditions.",
        "Sanitation and removal of infected or dead plant material are commonly used as part of black rot management.",
    ],
    "Apple Cedar Apple Rust": [
        "Cedar apple rust can produce yellow to orange leaf spots and may later develop darker structures on the underside of leaves.",
        "The disease has a life cycle involving apple and cedar or juniper hosts.",
        "Symptoms and disease development depend on environmental conditions and the presence of compatible hosts.",
    ],
    "Apple Healthy": [
        "Healthy apple leaves generally have an intact green surface without characteristic disease lesions.",
        "Uniform green tissue and absence of suspicious lesions support a healthy classification, although visual appearance alone is not a definitive diagnosis.",
    ],
}

retrieved_knowledge = LOCAL_KNOWLEDGE[pred_class]

print("=" * 70)
print("RETRIEVED KNOWLEDGE")
print("=" * 70)
for i, item in enumerate(retrieved_knowledge, 1):
    print(f"[{i}] {item}")


## Qwen grounded explanation

The following cells add the missing stage from the previous Notebook 8:

**RAG → Qwen**

Qwen is explicitly given:
- predicted class
- all class probabilities
- confidence
- entropy
- margin
- trust score/level
- Grad-CAM metadata
- retrieved disease knowledge

It is instructed to explain the specific prediction without treating Grad-CAM as proof of disease.


In [ ]:
# Cell 12 — Load Qwen2.5-7B-Instruct
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

try:
    import bitsandbytes  # noqa: F401
    BNB_AVAILABLE = True
except Exception:
    BNB_AVAILABLE = False

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the 7B Qwen demonstration.")

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)

if BNB_AVAILABLE:
    from transformers import BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    llm = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
else:
    print("bitsandbytes unavailable; attempting normal float16 loading.")
    llm = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
    )

llm.eval()
print("Qwen loaded:", QWEN_MODEL_NAME)


In [ ]:
# Cell 13 — Build grounded prompt
probability_text = "\n".join(
    f"- {CLASS_NAMES[i]}: {probs[i]*100:.2f}%"
    for i in range(len(CLASS_NAMES))
)

knowledge_text = "\n".join(
    f"{i+1}. {item}" for i, item in enumerate(retrieved_knowledge)
)

system_prompt = """You are an explainable AI assistant for an Apple plant disease classifier.

The classifier has already produced the prediction. Do not classify the image yourself.
Explain why the classifier prediction is plausible using only the supplied evidence.

Rules:
- Do not claim Grad-CAM proves a disease.
- Describe Grad-CAM as image regions that influenced the model prediction.
- Connect visual evidence cautiously to retrieved disease knowledge.
- Discuss confidence, uncertainty, and trust honestly.
- Do not call a prediction infallible just because confidence is high.
- Return valid JSON only with these fields:
  summary
  prediction_explanation
  visual_evidence
  knowledge_evidence
  uncertainty_note
  human_review_recommendation
  source_references
"""

user_prompt = f"""Explain this specific model prediction.

MODEL PREDICTION
Prediction: {pred_class}

CLASS PROBABILITIES
{probability_text}

MODEL METRICS
Confidence: {confidence:.4f}
Normalized entropy: {entropy:.4f}
Margin: {margin:.4f}
Trust score: {trust_score:.4f}
Trust level: {trust_level}
Human review required: {human_review_required}

GRAD-CAM
Target layer: ResNet50 layer4[-1]
High activation area: {high_activation_area:.2f}%
Grad-CAM identifies influential regions; it does not independently identify a disease.

RETRIEVED KNOWLEDGE
{knowledge_text}

Explain the prediction specifically for this image and these model outputs.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print(prompt_text[:5000])


In [ ]:
# Cell 14 — Generate Qwen explanation
inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    truncation=True,
    max_length=4096,
)
inputs = {k: v.to(llm.device) for k, v in inputs.items()}

with torch.no_grad():
    generated = llm.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = generated[0][inputs["input_ids"].shape[1]:]
raw_llm_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("=" * 70)
print("RAW QWEN OUTPUT")
print("=" * 70)
print(raw_llm_text)


In [ ]:
# Cell 15 — Parse Qwen JSON
def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass
    return None

llm_explanation = extract_json_object(raw_llm_text)

required_fields = [
    "summary",
    "prediction_explanation",
    "visual_evidence",
    "knowledge_evidence",
    "uncertainty_note",
    "human_review_recommendation",
    "source_references",
]

if llm_explanation is None:
    llm_explanation = {
        "summary": raw_llm_text,
        "prediction_explanation": raw_llm_text,
        "visual_evidence": "",
        "knowledge_evidence": "",
        "uncertainty_note": "",
        "human_review_recommendation": "Review according to the configured trust thresholds.",
        "source_references": [],
    }

for field in required_fields:
    llm_explanation.setdefault(field, "")

print("Qwen explanation fields validated.")


In [ ]:
# Cell 16 — Display the complete XAI + LLM result
print("\n" + "=" * 70)
print("FINAL XAI + LLM INFERENCE")
print("=" * 70)

print("\nPREDICTION")
print("-" * 70)
print("Prediction:", pred_class)
print(f"Confidence: {confidence*100:.2f}%")
print(f"Entropy:    {entropy:.4f}")
print(f"Margin:     {margin:.4f}")
print(f"Trust:      {trust_score:.4f} ({trust_level})")
print("Human review:", "REQUIRED" if human_review_required else "NOT REQUIRED")

print("\nLLM GROUNDED EXPLANATION")
print("-" * 70)

labels = [
    ("Summary", "summary"),
    ("Why the model predicted this class", "prediction_explanation"),
    ("Visual evidence from Grad-CAM", "visual_evidence"),
    ("Knowledge evidence from RAG", "knowledge_evidence"),
    ("Uncertainty note", "uncertainty_note"),
    ("Human-review recommendation", "human_review_recommendation"),
    ("Source references", "source_references"),
]

for title, key in labels:
    print(f"\n{title}:")
    value = llm_explanation.get(key, "")
    if isinstance(value, list):
        for item in value:
            print(" -", item)
    else:
        print(value)


In [ ]:
# Cell 17 — Save integrated inference and LLM explanation
inference_path = FINAL_DIR / "latest_inference_result.json"
llm_path = FINAL_DIR / "latest_llm_explanation.json"

inference_result = {
    "timestamp": datetime.now().isoformat(),
    "image_path": str(IMAGE_PATH),
    "prediction": {
        "label_id": pred_idx,
        "class": pred_class,
        "probabilities": {
            CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))
        },
        "confidence": confidence,
        "entropy": entropy,
        "margin": margin,
    },
    "trust": {
        "score": trust_score,
        "level": trust_level,
        "human_review_required": bool(human_review_required),
    },
    "gradcam": {
        "target_layer": "layer4[-1]",
        "high_activation_area_percent": high_activation_area,
        "path": str(gradcam_path),
    },
    "rag": {
        "knowledge_items": retrieved_knowledge,
    },
    "llm": {
        "model": QWEN_MODEL_NAME,
        "role": "grounded explanation only",
        "explanation": llm_explanation,
    },
    "human_feedback": {
        "status": "pending",
    },
}

inference_path.write_text(json.dumps(inference_result, indent=2, ensure_ascii=False))
llm_path.write_text(json.dumps(llm_explanation, indent=2, ensure_ascii=False))

print("Saved:", inference_path)
print("Saved:", llm_path)


# Human-in-the-Loop demonstration

The reviewer can enter:
- `accept` — agrees with the AI prediction
- `correct` — provides another class
- `reject` — rejects the AI output

The decision and optional comment are persisted to CSV and JSONL feedback memory.
If your teacher is present, they can enter the decision themselves during the demonstration.


In [ ]:
# Cell 19 — Capture real human feedback
print("=" * 70)
print("HUMAN-IN-THE-LOOP REVIEW")
print("=" * 70)
print("AI prediction:", pred_class)
print("Trust:", f"{trust_score:.4f}", f"({trust_level})")
print("Automatic review flag:", "REQUIRED" if human_review_required else "NOT REQUIRED")

decision = input(
    "\nEnter human decision [accept / correct / reject] "
    "(Enter = accept): "
).strip().lower()

if decision == "":
    decision = "accept"

if decision not in {"accept", "correct", "reject"}:
    raise ValueError("Decision must be accept, correct, or reject.")

corrected_label = ""
if decision == "correct":
    print("\nAvailable labels:")
    for i, name in enumerate(CLASS_NAMES):
        print(f"{i}: {name}")

    corrected_label = input("Enter corrected label exactly: ").strip()
    if corrected_label not in CLASS_NAMES:
        raise ValueError("Corrected label must match one of the available labels.")

comment = input("Reviewer comment (optional): ").strip()

reviewer_source = input(
    "Reviewer source [expert / annotator / user] (Enter = expert): "
).strip().lower()

if reviewer_source not in {"expert", "annotator", "user"}:
    reviewer_source = "expert"

print("\nCaptured:")
print("Decision:", decision)
print("Corrected label:", corrected_label or "N/A")
print("Comment:", comment or "N/A")
print("Source:", reviewer_source)


In [ ]:
# Cell 20 — Persist feedback to records and memory
feedback_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

feedback_record = {
    "feedback_id": feedback_id,
    "timestamp": datetime.now().isoformat(),
    "image_path": str(IMAGE_PATH),
    "predicted_label": pred_class,
    "predicted_label_id": pred_idx,
    "confidence": confidence,
    "entropy": entropy,
    "margin": margin,
    "trust_score": trust_score,
    "trust_level": trust_level,
    "human_review_required": bool(human_review_required),
    "human_decision": decision,
    "corrected_label": corrected_label,
    "reviewer_comment": comment,
    "feedback_source": reviewer_source,
    "llm_summary": llm_explanation.get("summary", ""),
}

records_csv = FEEDBACK_DIR / "feedback_records.csv"
records_jsonl = FEEDBACK_DIR / "feedback_records.jsonl"
memory_csv = FEEDBACK_DIR / "feedback_memory.csv"
memory_jsonl = FEEDBACK_DIR / "feedback_memory.jsonl"

new_df = pd.DataFrame([feedback_record])

if records_csv.exists():
    old_df = pd.read_csv(records_csv)
    all_records = pd.concat([old_df, new_df], ignore_index=True)
else:
    all_records = new_df

all_records.to_csv(records_csv, index=False)

with records_jsonl.open("a", encoding="utf-8") as f:
    f.write(json.dumps(feedback_record, ensure_ascii=False) + "\n")

all_records.to_csv(memory_csv, index=False)

with memory_jsonl.open("w", encoding="utf-8") as f:
    for row in all_records.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

inference_result["human_feedback"] = {
    "status": "completed",
    "feedback_id": feedback_id,
    "decision": decision,
    "corrected_label": corrected_label,
    "feedback_source": reviewer_source,
    "records_csv": str(records_csv),
    "memory_csv": str(memory_csv),
}

inference_path.write_text(json.dumps(inference_result, indent=2, ensure_ascii=False))

print("=" * 70)
print("FEEDBACK STORED")
print("=" * 70)
print("Feedback ID:", feedback_id)
print("Records CSV:", records_csv)
print("Records JSONL:", records_jsonl)
print("Memory CSV:", memory_csv)
print("Memory JSONL:", memory_jsonl)


In [ ]:
# Cell 21 — Evidence for teacher presentation
feedback_df = pd.read_csv(records_csv)

print("=" * 70)
print("HUMAN-IN-THE-LOOP EVIDENCE")
print("=" * 70)

display(feedback_df.tail(1).T)

print("\nStatistics")
print("Total reviewed:", len(feedback_df))
print("Accepted:", int((feedback_df["human_decision"] == "accept").sum()))
print("Corrected:", int((feedback_df["human_decision"] == "correct").sum()))
print("Rejected:", int((feedback_df["human_decision"] == "reject").sum()))

print("\nPersistent feedback memory: ✓")
print("Latest source:", feedback_df.iloc[-1]["feedback_source"])


In [ ]:
# Cell 22 — Final presentation dashboard
fig = plt.figure(figsize=(16, 10))

ax1 = fig.add_axes([0.03, 0.53, 0.45, 0.42])
ax1.imshow(image)
ax1.axis("off")
ax1.set_title(f"Input Image\nPrediction: {pred_class} | Confidence: {confidence*100:.2f}%")

ax2 = fig.add_axes([0.52, 0.53, 0.45, 0.42])
ax2.imshow(image)
ax2.imshow(cam, cmap="jet", alpha=0.45)
ax2.axis("off")
ax2.set_title("Grad-CAM — Influential Regions")

ax3 = fig.add_axes([0.03, 0.05, 0.94, 0.38])
ax3.axis("off")

dashboard_text = (
    "RESNET50 + TRUST\n"
    f"Prediction: {pred_class}\n"
    f"Confidence: {confidence*100:.2f}% | Entropy: {entropy:.4f} | Margin: {margin:.4f}\n"
    f"Trust: {trust_score:.4f} ({trust_level})\n"
    f"Human review: {'REQUIRED' if human_review_required else 'NOT REQUIRED'}\n\n"
    "QWEN GROUNDED EXPLANATION\n"
    f"{llm_explanation.get('summary', '')}\n\n"
    "HUMAN FEEDBACK\n"
    f"Decision: {decision.upper()} | Corrected label: {corrected_label or 'N/A'}\n"
    "Feedback persisted: YES"
)

ax3.text(0.01, 0.98, dashboard_text, va="top", ha="left", fontsize=11, wrap=True)
plt.show()

dashboard_path = FINAL_DIR / "latest_xai_hitl_dashboard.png"
fig.savefig(dashboard_path, dpi=160, bbox_inches="tight")
print("Dashboard saved:", dashboard_path)


In [ ]:
# Cell 23 — Final manifest
manifest = {
    "notebook": "08_Final_Integration_XAI_Dashboard_WITH_LLM_HITL",
    "timestamp": datetime.now().isoformat(),
    "classifier": "ResNet50",
    "demo_cases": DEMO_CASES,
    "active_demo_case": active_case["demo_case"],
    "active_image_index": DEMO_IMAGE_INDEX,
    "four_case_comparison_csv": str(FINAL_DIR / "four_trust_demo_comparison.csv"),
    "classes": CLASS_NAMES,
    "gradcam_layer": "layer4[-1]",
    "rag": True,
    "llm": {
        "model": QWEN_MODEL_NAME,
        "role": "grounded explanation only",
        "live_demo_generated": True,
        "notebook6_coverage": "30-sample demonstration/evaluation subset",
    },
    "human_in_the_loop": {
        "enabled": True,
        "decision": decision,
        "feedback_persisted": True,
        "feedback_source": reviewer_source,
        "model_retraining_from_feedback": False,
    },
    "outputs": [
        str(inference_path),
        str(llm_path),
        str(gradcam_path),
        str(dashboard_path),
        str(records_csv),
        str(records_jsonl),
        str(memory_csv),
        str(memory_jsonl),
    ],
}

comparison_path = FINAL_DIR / "four_trust_demo_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

manifest_path = FINAL_DIR / "notebook8_final_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

print("=" * 70)
print("NOTEBOOK 8 COMPLETE")
print("=" * 70)
print(json.dumps(manifest, indent=2))


# Final Documentation

## Complete architecture

```text
Apple Leaf Image
      ↓
ResNet50 Classifier
      ↓
Prediction + Probabilities
      ↓
Trust / Uncertainty
      ↓
Grad-CAM (layer4[-1])
      ↓
RAG Disease Knowledge
      ↓
Qwen2.5-7B-Instruct
      ↓
Grounded Explanation
      ↓
Human Review
      ↓
Persistent Feedback Memory
```

## What can be demonstrated to the teacher

1. **Prediction:** ResNet50 predicts one of four Apple classes.
2. **Explainability:** Grad-CAM shows regions that influenced the prediction.
3. **Uncertainty:** confidence, entropy, margin and trust are reported.
4. **RAG:** disease knowledge is retrieved and supplied to Qwen.
5. **LLM:** Qwen explains the specific model prediction using the supplied evidence.
6. **Human-in-the-Loop:** reviewer accepts, corrects or rejects the output.
7. **Feedback memory:** the review is saved to CSV and JSONL.

## Scientifically accurate claim

The current project stores human feedback as persistent feedback memory. It does **not** automatically retrain ResNet50 from that feedback. Therefore do not claim that the classifier has learned from the feedback unless a separate retraining stage is implemented.

## LLM coverage

Notebook 6 generated Qwen explanations for a 30-sample demonstration/evaluation subset to control runtime. Notebook 8 additionally generates a fresh Qwen explanation for the selected live demonstration image.

## Main outputs

```text
XAI_Project/results/final_integration/
├── latest_inference_result.json
├── latest_llm_explanation.json
├── latest_gradcam.png
├── latest_xai_hitl_dashboard.png
└── notebook8_final_manifest.json

XAI_Project/results/human_feedback/
├── feedback_records.csv
├── feedback_records.jsonl
├── feedback_memory.csv
└── feedback_memory.jsonl
```
